# 03 — Data Cleaning And Preprocessing

**Pipeline role.** Third notebook. Converts raw data into a clean, split, modeling-ready form and locks in preprocessing policy before feature engineering.

**Rubric coverage (from `Group Project Guideline.md`).**
- Report Section 3 — Model Building (preprocessing subsection): cleaning, missing-value handling, encoding, scaling, train/test split.
- Supports Section 4 — Performance Evaluation by freezing a held-out test set used only in notebook 07.

**TL;DR for teammates.** Every cleaning decision made here must be justified and reproducible. Fit-on-train/apply-on-test discipline starts here. The outputs of this notebook (cleaned train/test splits) are the single source of truth for notebooks 04–07.


## Inputs, Outputs, Artifacts

**Inputs.**
- `data/raw/hotel_bookings.csv`.
- Leakage list, missingness table, outlier notes from notebook 02.
- [../references/project_conventions.md](../references/project_conventions.md) (leakage and reproducibility rules).

**Outputs.**
- `data/interim/hotel_cleaned_v1.parquet` (or CSV) — cleaned full dataset before split.
- `data/processed/X_train_v1.parquet`, `y_train_v1.parquet`, `X_test_v1.parquet`, `y_test_v1.parquet` — frozen stratified 80/20 split, `random_state=42`.
- Preprocessing policy notes (in this notebook) documenting decisions: dropped columns, imputation strategy, encoding strategy, scaling strategy.
- Optional fitted preprocessor object stored in-notebook for consistent reuse in notebook 05/06/07.

**Downstream consumers.**
- Notebook 04 reads cleaned data to engineer features.
- Notebooks 05/06 use the processed splits for modeling and tuning.
- Notebook 07 uses the frozen test split exactly once for final evaluation.


## Methodological Influences

Cleaning → missing-value policy → encoding → scaling → stratified split follows standard ISOM3360 preprocessing practice with fit-on-train / apply-on-test discipline. See the shared [project conventions](../references/project_conventions.md) for standards applied across all notebooks.


## Key Questions To Answer Here

1. Which columns are dropped and why?
2. How are missing values handled, column by column?
3. How are implausible rows (e.g., zero-guest bookings, negative `adr`) handled?
4. How are categorical variables encoded, and where is the fit/apply boundary?
5. How are numeric variables scaled, and for which models?
6. What is the exact train/test split protocol?
7. What alternatives were considered and rejected, and why?


## Work Plan

### 3.1 Setup And Load
- Import pandas, numpy, scikit-learn (`train_test_split`, `SimpleImputer`, `OneHotEncoder`, `StandardScaler`, `ColumnTransformer`, `Pipeline`).
- Load `data/raw/hotel_bookings.csv`.
- Re-assert shape and target name.

### 3.2 Drop Leakage Columns
- Drop `reservation_status`, `reservation_status_date`.
- TODO: decide on `assigned_room_type` and `booking_changes` based on NB02 findings.
- Record the final excluded list in this notebook and in [../references/data_dictionary_template.md](../references/data_dictionary_template.md).

### 3.3 Duplicates
- Report number of exact duplicate rows.
- Decide whether to drop (usually yes) and document.

### 3.4 Missing Value Policy
- Per-column strategy, informed by NB02:
  - `country`: impute with `"Unknown"` or mode; high-cardinality handling deferred to NB04.
  - `agent`, `company`: impute with `0` or `"None"` (they represent "no agent/company"); document choice.
  - `children`: impute with median (typically 0).
  - Any other column flagged in NB02: document decision.
- Fit imputers on training fold only.

### 3.5 Type Corrections And Implausible Values
- Convert dtypes where needed.
- Rule for zero-guest bookings: drop or keep with flag (document).
- Rule for `adr` outliers (negative or extreme): clip or drop (document and log the threshold).

### 3.6 Encoding Policy (Scaffold Only)
- Low-cardinality categoricals → one-hot.
- High-cardinality categoricals (`country`, `agent`, `company`) → target/frequency encoding or bucket-then-one-hot; final choice can move to NB04.
- Note: encoding is *fit* here and *applied* everywhere downstream.

### 3.7 Scaling Policy (Scaffold Only)
- Standardize numeric features for scale-sensitive models (LR, k-NN).
- Tree-based models can skip scaling; keep both views consistent by using a `ColumnTransformer`.

### 3.8 Train/Test Split
- Stratified 80/20 on `is_canceled`, `random_state=42`.
- Save `X_train`, `X_test`, `y_train`, `y_test` to `data/processed/` (versioned).
- Confirm target proportion in both splits.

### 3.9 Alternative Decisions Log
- Record any preprocessing alternative considered but rejected (e.g., median vs mode imputation for `country`) and the one-line rationale. This makes the report §3 narrative concrete.


## Implementation Block 3.1 — Setup And Load

**Scope.** Section 3.1 only — import the preprocessing dependencies, load the raw hotel-bookings dataset, and confirm the expected intake assumptions before any cleaning rules are applied.

The purpose is to establish a stable starting point for all later preprocessing steps in this notebook. No cleaning, dropping, imputation, encoding, or splitting is performed yet.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RAW_DATA_CANDIDATES = [
    Path("../data/raw/hotel_bookings.csv"),
    Path("data/raw/hotel_bookings.csv"),
]
EXPECTED_SHAPE = (119390, 32)
TARGET_COLUMN = "is_canceled"

RAW_DATA_PATH = next((path for path in RAW_DATA_CANDIDATES if path.exists()), RAW_DATA_CANDIDATES[0])

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError("hotel_bookings.csv was not found in the expected raw-data locations.")

df = pd.read_csv(RAW_DATA_PATH)

if df.shape != EXPECTED_SHAPE:
    raise ValueError(f"Expected dataset shape {EXPECTED_SHAPE}, found {df.shape}.")

if TARGET_COLUMN not in df.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' is missing from the raw dataset.")

setup_and_load_summary = pd.DataFrame(
    {
        "check": [
            "resolved_raw_data_path",
            "dataset_shape",
            "target_column_present",
            "feature_count_excluding_target",
            "duplicate_row_count_initial",
        ],
        "value": [
            str(RAW_DATA_PATH.resolve()),
            str(df.shape),
            TARGET_COLUMN in df.columns,
            df.shape[1] - 1,
            int(df.duplicated().sum()),
        ],
    }
)

display(setup_and_load_summary)
display(df.head())


,check,value
0,resolved_raw_data_path,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,dataset_shape,"(119390, 32)"
2,target_column_present,True
3,feature_count_excluding_target,31
4,duplicate_row_count_initial,31994


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## Implementation Block 3.2 — Drop Leakage Columns

**Scope.** Section 3.2 only — remove columns that are already confirmed as leakage risks, and explicitly record the columns that still require timing-based review before a final keep/drop decision is made.

Based on Notebook 02, `reservation_status` and `reservation_status_date` are treated as confirmed leakage exclusions because they directly reflect post-booking outcomes. `assigned_room_type` and `booking_changes` are not dropped in this step; they are carried forward temporarily and flagged for explicit timing review later in this notebook.

The purpose of this block is to create a modeling-safe working copy of the dataset before duplicate handling, missing-value treatment, and split logic are introduced.


In [2]:
confirmed_leakage_columns = [
    column
    for column in ["reservation_status", "reservation_status_date"]
    if column in df.columns
]
review_required_columns = [
    column for column in ["assigned_room_type", "booking_changes"] if column in df.columns
]

preprocessed_df = df.drop(columns=confirmed_leakage_columns).copy()

leakage_policy_summary = pd.DataFrame(
    {
        "column": confirmed_leakage_columns + review_required_columns,
        "status": ["drop_now"] * len(confirmed_leakage_columns)
        + ["review_required_keep_temporarily"] * len(review_required_columns),
        "rationale": [
            "Post-outcome field that would leak booking resolution into the model.",
            "Post-outcome timestamp that would leak booking resolution timing into the model.",
            "Potentially usable only if its timestamp is confirmed to be known before cancellation outcome.",
            "Potentially usable only if its update timing is confirmed to be known before cancellation outcome.",
        ][: len(confirmed_leakage_columns) + len(review_required_columns)],
    }
)

column_count_summary = pd.DataFrame(
    {
        "check": [
            "original_column_count",
            "column_count_after_confirmed_leakage_drop",
            "confirmed_leakage_columns_dropped",
            "review_required_columns_retained_for_now",
        ],
        "value": [
            df.shape[1],
            preprocessed_df.shape[1],
            ", ".join(confirmed_leakage_columns),
            ", ".join(review_required_columns),
        ],
    }
)

display(leakage_policy_summary)
display(column_count_summary)


,column,status,rationale
0,reservation_status,drop_now,Post-outcome field that would leak booking res...
1,reservation_status_date,drop_now,Post-outcome timestamp that would leak booking...
2,assigned_room_type,review_required_keep_temporarily,Potentially usable only if its timestamp is co...
3,booking_changes,review_required_keep_temporarily,Potentially usable only if its update timing i...


,check,value
0,original_column_count,32
1,column_count_after_confirmed_leakage_drop,30
2,confirmed_leakage_columns_dropped,"reservation_status, reservation_status_date"
3,review_required_columns_retained_for_now,"assigned_room_type, booking_changes"


## Implementation Block 3.3 — Duplicates

**Scope.** Section 3.3 only — measure exact duplicate rows after the confirmed leakage columns are removed, decide whether to drop them, and record the effect of that decision.

The current default policy is to drop exact duplicates because they do not add new information and can distort downstream split counts and model evaluation. This block keeps the decision explicit by reporting the duplicate count before and after deduplication.

The purpose of this step is to produce a cleaner working dataset before missing-value handling and type corrections are applied.


In [3]:
duplicate_row_count_after_leakage_drop = int(preprocessed_df.duplicated().sum())

preprocessed_deduplicated_df = preprocessed_df.drop_duplicates().copy()
duplicate_rows_removed = preprocessed_df.shape[0] - preprocessed_deduplicated_df.shape[0]

duplicate_handling_summary = pd.DataFrame(
    {
        "check": [
            "row_count_before_deduplication",
            "exact_duplicate_row_count",
            "duplicate_policy",
            "rows_removed_by_deduplication",
            "row_count_after_deduplication",
        ],
        "value": [
            preprocessed_df.shape[0],
            duplicate_row_count_after_leakage_drop,
            "drop_exact_duplicates",
            duplicate_rows_removed,
            preprocessed_deduplicated_df.shape[0],
        ],
    }
)

sample_duplicate_rows = preprocessed_df.loc[preprocessed_df.duplicated(keep=False)].head(10)

display(duplicate_handling_summary)
display(sample_duplicate_rows)


,check,value
0,row_count_before_deduplication,119390
1,exact_duplicate_row_count,32252
2,duplicate_policy,drop_exact_duplicates
3,rows_removed_by_deduplication,32252
4,row_count_after_deduplication,87138


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,A,0,No Deposit,240.0,NaN,0,Transient,98.00,0,1
21,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,A,1,No Deposit,250.0,NaN,0,Transient,84.67,0,1
39,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1
43,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,E,0,No Deposit,250.0,NaN,0,Transient,137.00,0,1
132,Resort Hotel,1,5,2015,July,28,5,1,0,2,...,D,0,No Deposit,240.0,NaN,0,Transient,97.00,0,0
138,Resort Hotel,1,5,2015,July,28,5,1,0,2,...,D,0,No Deposit,240.0,NaN,0,Transient,97.00,0,0
198,Resort Hotel,0,0,2015,July,28,7,0,1,1,...,A,0,No Deposit,240.0,NaN,0,Transient,109.80,0,3
200,Resort Hotel,0,0,2015,July,28,7,0,1,1,...,A,0,No Deposit,240.0,NaN,0,Transient,109.80,0,3


## Implementation Block 3.4 — Missing Value Policy

**Scope.** Section 3.4 only — quantify the remaining missingness after confirmed leakage removal and deduplication, then record the planned column-by-column imputation policy.

To preserve fit-on-train/apply-on-test discipline, this block does not fit any imputer yet. Instead, it defines the imputation rules that will be applied only after the training split is created later in this notebook.

The purpose of this step is to make the preprocessing policy explicit before type corrections, implausible-value handling, and split logic are introduced.


In [4]:
missing_value_counts_after_dedup = (
    preprocessed_deduplicated_df.isna().sum().sort_values(ascending=False)
)
columns_with_missing_values_after_dedup = (
    missing_value_counts_after_dedup[missing_value_counts_after_dedup > 0]
    .rename_axis("column")
    .reset_index(name="missing_count")
)
columns_with_missing_values_after_dedup["missing_pct"] = (
    columns_with_missing_values_after_dedup["missing_count"]
    / preprocessed_deduplicated_df.shape[0]
    * 100
).round(2)

imputation_plan = {
    "company": {
        "planned_strategy": "constant",
        "planned_fill_value": 0,
        "note": "Treat missing company as no registered company identifier.",
    },
    "agent": {
        "planned_strategy": "constant",
        "planned_fill_value": 0,
        "note": "Treat missing agent as no registered agent identifier.",
    },
    "country": {
        "planned_strategy": "constant",
        "planned_fill_value": "Unknown",
        "note": "Preserve the row while making missing origin explicit.",
    },
    "children": {
        "planned_strategy": "median",
        "planned_fill_value": "fit_on_train_only",
        "note": "Small numeric missingness should be imputed from the training distribution only.",
    },
}

planned_imputers = {
    "company": SimpleImputer(strategy="constant", fill_value=0),
    "agent": SimpleImputer(strategy="constant", fill_value=0),
    "country": SimpleImputer(strategy="constant", fill_value="Unknown"),
    "children": SimpleImputer(strategy="median"),
}

missing_value_policy_summary = columns_with_missing_values_after_dedup.merge(
    pd.DataFrame.from_dict(imputation_plan, orient="index").reset_index(names="column"),
    on="column",
    how="left",
)
missing_value_policy_summary["fit_stage"] = "after_train_test_split_only"

display(columns_with_missing_values_after_dedup)
display(missing_value_policy_summary)


,column,missing_count,missing_pct
0,company,81890,93.98
1,agent,12160,13.95
2,country,451,0.52
3,children,4,0.00


,column,missing_count,missing_pct,planned_strategy,planned_fill_value,note,fit_stage
0,company,81890,93.98,constant,0,Treat missing company as no registered company...,after_train_test_split_only
1,agent,12160,13.95,constant,0,Treat missing agent as no registered agent ide...,after_train_test_split_only
2,country,451,0.52,constant,Unknown,Preserve the row while making missing origin e...,after_train_test_split_only
3,children,4,0.00,median,fit_on_train_only,Small numeric missingness should be imputed fr...,after_train_test_split_only


## Implementation Block 3.5 — Type Corrections And Implausible Values

**Scope.** Section 3.5 only — apply limited type corrections that are safe before model fitting, then document and execute the handling rules for implausible rows identified in Notebook 02.

The policy used here is intentionally conservative. `children` is converted to a nullable integer type because it represents a count field, zero-guest bookings are dropped because they do not represent a valid stay request, and negative `adr` values are clipped to 0 because they are economically implausible and rare in this dataset.

The purpose of this step is to create a cleaner working dataset for later split and transformation steps without fitting any downstream preprocessing objects yet.


In [5]:
preprocessed_validated_df = preprocessed_deduplicated_df.copy()

children_is_integer_like = (
    preprocessed_validated_df["children"].dropna() % 1 == 0
).all()
if not children_is_integer_like:
    raise ValueError("`children` contains non-integer-like values and cannot be safely cast to Int64.")

preprocessed_validated_df["children"] = preprocessed_validated_df["children"].astype("Int64")

preprocessed_validated_df["total_guests_proxy"] = (
    preprocessed_validated_df["adults"]
    + preprocessed_validated_df["babies"]
    + preprocessed_validated_df["children"].fillna(0).astype("Int64")
)
zero_guest_row_count = int((preprocessed_validated_df["total_guests_proxy"] == 0).sum())
negative_adr_row_count = int((preprocessed_validated_df["adr"] < 0).sum())
max_adr_before_correction = float(preprocessed_validated_df["adr"].max())

preprocessed_validated_df = preprocessed_validated_df.loc[
    preprocessed_validated_df["total_guests_proxy"] > 0
].copy()
preprocessed_validated_df["adr"] = preprocessed_validated_df["adr"].clip(lower=0)
preprocessed_validated_df = preprocessed_validated_df.drop(columns=["total_guests_proxy"])

implausible_value_handling_summary = pd.DataFrame(
    {
        "check": [
            "children_cast_to_nullable_int",
            "zero_guest_policy",
            "zero_guest_rows_removed",
            "negative_adr_policy",
            "negative_adr_rows_corrected",
            "max_adr_before_correction",
            "row_count_after_implausible_value_handling",
        ],
        "value": [
            True,
            "drop_rows_with_zero_total_guests",
            zero_guest_row_count,
            "clip_negative_adr_to_zero",
            negative_adr_row_count,
            round(max_adr_before_correction, 2),
            preprocessed_validated_df.shape[0],
        ],
    }
)

dtype_summary = pd.DataFrame(
    {
        "column": ["children", "agent", "company", "adr"],
        "dtype_after_step": [
            str(preprocessed_validated_df["children"].dtype),
            str(preprocessed_validated_df["agent"].dtype),
            str(preprocessed_validated_df["company"].dtype),
            str(preprocessed_validated_df["adr"].dtype),
        ],
        "note": [
            "Converted to nullable integer count type.",
            "Retained as numeric identifier until post-split imputation.",
            "Retained as numeric identifier until post-split imputation.",
            "Retained as numeric rate after clipping negative values.",
        ],
    }
)

display(implausible_value_handling_summary)
display(dtype_summary)


,check,value
0,children_cast_to_nullable_int,True
1,zero_guest_policy,drop_rows_with_zero_total_guests
2,zero_guest_rows_removed,166
3,negative_adr_policy,clip_negative_adr_to_zero
4,negative_adr_rows_corrected,1
5,max_adr_before_correction,5400.0
6,row_count_after_implausible_value_handling,86972


,column,dtype_after_step,note
0,children,Int64,Converted to nullable integer count type.
1,agent,float64,Retained as numeric identifier until post-spli...
2,company,float64,Retained as numeric identifier until post-spli...
3,adr,float64,Retained as numeric rate after clipping negati...


## Implementation Block 3.6 — Encoding Policy Scaffold

**Scope.** Section 3.6 only — identify the categorical fields that are ready for low-cardinality one-hot encoding, separate the fields that should remain under high-cardinality or timing review, and define a reusable encoder object without fitting it yet.

This is still a planning step rather than a fit step. The encoder is defined here so downstream preprocessing stays reproducible, but no transformation is fit until after the train/test split is created.

The purpose of this block is to make the encoding boundary explicit before scaling and split logic are introduced.


In [6]:
categorical_columns_for_encoding = preprocessed_validated_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

high_cardinality_threshold = 15
categorical_cardinality_summary = pd.DataFrame(
    {
        "column": categorical_columns_for_encoding,
        "unique_values": [
            preprocessed_validated_df[column].nunique(dropna=True)
            for column in categorical_columns_for_encoding
        ],
    }
).sort_values(by=["unique_values", "column"], ascending=[False, True]).reset_index(drop=True)

high_cardinality_categorical_columns = sorted(
    categorical_cardinality_summary.loc[
        categorical_cardinality_summary["unique_values"] > high_cardinality_threshold, "column"
    ].tolist()
)

explicit_deferred_encoding_columns = sorted(
    {
        *high_cardinality_categorical_columns,
        *[column for column in ["country", "agent", "company"] if column in preprocessed_validated_df.columns],
        *review_required_columns,
    }
)
low_cardinality_onehot_columns = sorted(
    [
        column
        for column in categorical_columns_for_encoding
        if column not in explicit_deferred_encoding_columns
    ]
)

encoding_policy_rows = []
for column in low_cardinality_onehot_columns:
    encoding_policy_rows.append(
        {
            "column": column,
            "planned_encoding": "one_hot",
            "fit_stage": "after_train_test_split_only",
            "note": "Low-cardinality categorical column ready for one-hot encoding.",
        }
    )
for column in explicit_deferred_encoding_columns:
    note = "Deferred for high-cardinality or timing review before final encoding choice."
    if column in review_required_columns:
        note = "Deferred until timing review confirms whether the column is modeling-safe."
    encoding_policy_rows.append(
        {
            "column": column,
            "planned_encoding": "defer_decision",
            "fit_stage": "after_train_test_split_only",
            "note": note,
        }
    )

encoding_policy_summary = pd.DataFrame(encoding_policy_rows).sort_values(
    by=["planned_encoding", "column"]
).reset_index(drop=True)

onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

encoding_scope_summary = pd.DataFrame(
    {
        "check": [
            "categorical_columns_detected",
            "low_cardinality_onehot_columns",
            "deferred_encoding_columns",
            "onehot_handle_unknown_policy",
        ],
        "value": [
            len(categorical_columns_for_encoding),
            ", ".join(low_cardinality_onehot_columns),
            ", ".join(explicit_deferred_encoding_columns),
            "ignore",
        ],
    }
)

display(categorical_cardinality_summary)
display(encoding_policy_summary)
display(encoding_scope_summary)


,column,unique_values
0,country,177
1,arrival_date_month,12
2,assigned_room_type,11
3,reserved_room_type,9
4,market_segment,8
5,distribution_channel,5
6,meal,5
7,customer_type,4
8,deposit_type,3
9,hotel,2


,column,planned_encoding,fit_stage,note
0,agent,defer_decision,after_train_test_split_only,Deferred for high-cardinality or timing review...
1,assigned_room_type,defer_decision,after_train_test_split_only,Deferred until timing review confirms whether ...
2,booking_changes,defer_decision,after_train_test_split_only,Deferred until timing review confirms whether ...
3,company,defer_decision,after_train_test_split_only,Deferred for high-cardinality or timing review...
4,country,defer_decision,after_train_test_split_only,Deferred for high-cardinality or timing review...
5,arrival_date_month,one_hot,after_train_test_split_only,Low-cardinality categorical column ready for o...
6,customer_type,one_hot,after_train_test_split_only,Low-cardinality categorical column ready for o...
7,deposit_type,one_hot,after_train_test_split_only,Low-cardinality categorical column ready for o...
8,distribution_channel,one_hot,after_train_test_split_only,Low-cardinality categorical column ready for o...
9,hotel,one_hot,after_train_test_split_only,Low-cardinality categorical column ready for o...


,check,value
0,categorical_columns_detected,10
1,low_cardinality_onehot_columns,"arrival_date_month, customer_type, deposit_typ..."
2,deferred_encoding_columns,"agent, assigned_room_type, booking_changes, co..."
3,onehot_handle_unknown_policy,ignore


## Implementation Block 3.7 — Scaling Policy Scaffold

**Scope.** Section 3.7 only — identify the numeric features that are candidates for scaling, separate them from deferred identifier-style fields, and define reusable scaling scaffolds without fitting them yet.

The policy here is model-aware. A scaled numeric view is prepared for scale-sensitive models such as logistic regression and k-NN, while a passthrough view remains available for tree-based models that do not require scaling.

The purpose of this block is to lock in the scaling boundary before the train/test split and artifact-saving steps are introduced.


In [7]:
numeric_columns_for_modeling = preprocessed_validated_df.select_dtypes(
    include=[np.number, "Int64"]
).columns.tolist()

excluded_numeric_columns = sorted(
    {
        TARGET_COLUMN,
        "agent",
        "company",
        *review_required_columns,
    }
    & set(numeric_columns_for_modeling)
)
scalable_numeric_columns = sorted(
    [column for column in numeric_columns_for_modeling if column not in excluded_numeric_columns]
)

scaling_policy_summary = pd.DataFrame(
    {
        "column": scalable_numeric_columns + excluded_numeric_columns,
        "planned_scaling": ["standardize"] * len(scalable_numeric_columns)
        + ["exclude_for_now"] * len(excluded_numeric_columns),
        "fit_stage": ["after_train_test_split_only"]
        * (len(scalable_numeric_columns) + len(excluded_numeric_columns)),
        "note": [
            "Numeric feature planned for StandardScaler in scale-sensitive model pipelines."
            for _ in scalable_numeric_columns
        ]
        + [
            "Temporarily excluded from scaling because it is the target, a deferred identifier field, or under timing review."
            for _ in excluded_numeric_columns
        ],
    }
)

numeric_scaler = StandardScaler()
scaled_numeric_preprocessor = ColumnTransformer(
    transformers=[("numeric", numeric_scaler, scalable_numeric_columns)],
    remainder="drop",
)
scaled_numeric_pipeline = Pipeline(
    steps=[("preprocessor", scaled_numeric_preprocessor)]
)

scaling_scope_summary = pd.DataFrame(
    {
        "check": [
            "numeric_columns_detected",
            "scalable_numeric_columns",
            "excluded_numeric_columns",
            "tree_model_view",
            "scale_sensitive_model_view",
        ],
        "value": [
            len(numeric_columns_for_modeling),
            ", ".join(scalable_numeric_columns),
            ", ".join(excluded_numeric_columns),
            "numeric passthrough remains available",
            "StandardScaler scaffold defined but not fitted",
        ],
    }
)

display(scaling_policy_summary)
display(scaling_scope_summary)


,column,planned_scaling,fit_stage,note
0,adr,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
1,adults,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
2,arrival_date_day_of_month,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
3,arrival_date_week_number,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
4,arrival_date_year,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
5,babies,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
6,children,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
7,days_in_waiting_list,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
8,is_repeated_guest,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...
9,lead_time,standardize,after_train_test_split_only,Numeric feature planned for StandardScaler in ...


,check,value
0,numeric_columns_detected,20
1,scalable_numeric_columns,"adr, adults, arrival_date_day_of_month, arriva..."
2,excluded_numeric_columns,"agent, booking_changes, company, is_canceled"
3,tree_model_view,numeric passthrough remains available
4,scale_sensitive_model_view,StandardScaler scaffold defined but not fitted


## Implementation Block 3.8 — Train/Test Split And Artifact Save

**Scope.** Section 3.8 only — create the frozen stratified 80/20 split using `random_state=42`, save the cleaned artifacts to the project data folders, and verify that the target balance is preserved across the overall dataset, training split, and test split.

This split is performed after the dataset has been cleaned for confirmed leakage, duplicates, and implausible values, but before any imputer, encoder, or scaler is fitted. That keeps the fit-on-train/apply-on-test boundary intact for all downstream preprocessing.

The purpose of this step is to create the single source of truth artifacts that downstream notebooks can reuse consistently.


In [8]:
ARTIFACT_VERSION = "v1"
RANDOM_STATE = 42
TEST_SIZE = 0.2

interim_dir = Path("../data/interim")
processed_dir = Path("../data/processed")
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

modeling_dataset = preprocessed_validated_df.copy()
X = modeling_dataset.drop(columns=[TARGET_COLUMN])
y = modeling_dataset[TARGET_COLUMN].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

y_train = y_train.rename(TARGET_COLUMN)
y_test = y_test.rename(TARGET_COLUMN)

cleaned_full_dataset_path = interim_dir / f"hotel_cleaned_{ARTIFACT_VERSION}.csv"
X_train_path = processed_dir / f"X_train_{ARTIFACT_VERSION}.csv"
X_test_path = processed_dir / f"X_test_{ARTIFACT_VERSION}.csv"
y_train_path = processed_dir / f"y_train_{ARTIFACT_VERSION}.csv"
y_test_path = processed_dir / f"y_test_{ARTIFACT_VERSION}.csv"

modeling_dataset.to_csv(cleaned_full_dataset_path, index=False)
X_train.to_csv(X_train_path, index=False)
X_test.to_csv(X_test_path, index=False)
y_train.to_frame().to_csv(y_train_path, index=False)
y_test.to_frame().to_csv(y_test_path, index=False)

split_balance_summary = pd.DataFrame(
    {
        "dataset": ["overall", "train", "test"],
        "rows": [
            modeling_dataset.shape[0],
            X_train.shape[0],
            X_test.shape[0],
        ],
        "cancellation_rate_pct": [
            round(y.mean() * 100, 2),
            round(y_train.mean() * 100, 2),
            round(y_test.mean() * 100, 2),
        ],
    }
)

artifact_save_summary = pd.DataFrame(
    {
        "artifact": [
            "cleaned_full_dataset",
            "X_train",
            "X_test",
            "y_train",
            "y_test",
        ],
        "path": [
            str(cleaned_full_dataset_path.resolve()),
            str(X_train_path.resolve()),
            str(X_test_path.resolve()),
            str(y_train_path.resolve()),
            str(y_test_path.resolve()),
        ],
    }
)

display(split_balance_summary)
display(artifact_save_summary)


,dataset,rows,cancellation_rate_pct
0,overall,86972,27.31
1,train,69577,27.31
2,test,17395,27.31


,artifact,path
0,cleaned_full_dataset,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
1,X_train,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
2,X_test,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
3,y_train,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...
4,y_test,/Users/y.h.lam/Documents/GitHub/ISOM3360-Proje...


## Implementation Block 3.9 — Alternative Decisions Log

**Scope.** Section 3.9 only — record the main preprocessing alternatives considered during Notebook 03, together with the final decision and a concise rationale.

The purpose of this block is to leave a report-ready audit trail for Section 3 of the project write-up. It captures not only what was done, but also what was considered and rejected.


In [9]:
alternative_decisions_log = pd.DataFrame(
    [
        {
            "topic": "confirmed leakage columns",
            "alternatives_considered": "Keep `reservation_status` and `reservation_status_date` as predictive features.",
            "final_decision": "Drop both columns before modeling.",
            "rationale": "They encode post-outcome information and would leak cancellation resolution into the model.",
        },
        {
            "topic": "review-required timing fields",
            "alternatives_considered": "Drop `assigned_room_type` and `booking_changes` immediately.",
            "final_decision": "Retain temporarily and defer final keep/drop decision.",
            "rationale": "These fields may still be usable, but only if their timestamps can be justified as pre-outcome.",
        },
        {
            "topic": "duplicate rows",
            "alternatives_considered": "Keep exact duplicate bookings to preserve raw row count.",
            "final_decision": "Drop exact duplicates.",
            "rationale": "Exact duplicates add no new information and can distort model fitting and evaluation counts.",
        },
        {
            "topic": "missing `country` values",
            "alternatives_considered": "Impute with mode instead of an explicit placeholder category.",
            "final_decision": "Plan constant fill with `Unknown` after the split.",
            "rationale": "Using an explicit label preserves the uncertainty rather than forcing a possibly misleading dominant country.",
        },
        {
            "topic": "missing `agent` and `company` values",
            "alternatives_considered": "Drop the columns or treat all missing values as unusable noise.",
            "final_decision": "Plan constant fill with 0 after the split.",
            "rationale": "The missing pattern plausibly reflects no registered agent or company, so a sentinel value is more faithful than dropping the rows.",
        },
        {
            "topic": "zero-guest bookings",
            "alternatives_considered": "Keep the rows and add a flag instead of removing them.",
            "final_decision": "Drop rows with zero total guests.",
            "rationale": "A booking with no adults, children, or babies is not a valid stay request and is better treated as implausible data.",
        },
        {
            "topic": "negative `adr` values",
            "alternatives_considered": "Drop the affected row or leave the value untouched.",
            "final_decision": "Clip negative `adr` to 0.",
            "rationale": "The anomaly is extremely rare and clipping preserves the row while enforcing a non-negative rate.",
        },
        {
            "topic": "categorical encoding strategy",
            "alternatives_considered": "One-hot encode all categorical variables immediately.",
            "final_decision": "One-hot only low-cardinality columns and defer high-cardinality or timing-sensitive fields.",
            "rationale": "This avoids premature expansion and leaves room for more appropriate handling of `country`, `agent`, `company`, and timing-review fields.",
        },
        {
            "topic": "numeric scaling strategy",
            "alternatives_considered": "Apply scaling to every numeric field, including identifiers and the target.",
            "final_decision": "Scale only modeling-ready numeric predictors and exclude the target and deferred identifier-style fields.",
            "rationale": "Scaling should support scale-sensitive models without contaminating target handling or unresolved identifier fields.",
        },
    ]
)

display(alternative_decisions_log)


,topic,alternatives_considered,final_decision,rationale
0,confirmed leakage columns,Keep `reservation_status` and `reservation_sta...,Drop both columns before modeling.,They encode post-outcome information and would...
1,review-required timing fields,Drop `assigned_room_type` and `booking_changes...,Retain temporarily and defer final keep/drop d...,"These fields may still be usable, but only if ..."
2,duplicate rows,Keep exact duplicate bookings to preserve raw ...,Drop exact duplicates.,Exact duplicates add no new information and ca...
3,missing `country` values,Impute with mode instead of an explicit placeh...,Plan constant fill with `Unknown` after the sp...,Using an explicit label preserves the uncertai...
4,missing `agent` and `company` values,Drop the columns or treat all missing values a...,Plan constant fill with 0 after the split.,The missing pattern plausibly reflects no regi...
5,zero-guest bookings,Keep the rows and add a flag instead of removi...,Drop rows with zero total guests.,"A booking with no adults, children, or babies ..."
6,negative `adr` values,Drop the affected row or leave the value untou...,Clip negative `adr` to 0.,The anomaly is extremely rare and clipping pre...
7,categorical encoding strategy,One-hot encode all categorical variables immed...,One-hot only low-cardinality columns and defer...,This avoids premature expansion and leaves roo...
8,numeric scaling strategy,"Apply scaling to every numeric field, includin...",Scale only modeling-ready numeric predictors a...,Scaling should support scale-sensitive models ...


## Review Checklist

- [x] Leakage columns (`reservation_status`, `reservation_status_date`, plus any confirmed in NB02) are explicitly dropped with rationale.
- [x] Duplicate rows handled (count reported; decision justified).
- [x] Missing values handled per-column with documented strategy (drop / median / mode / "Undefined" / indicator).
- [x] Type corrections applied where needed (e.g., `children` to int, `reservation_status_date` to datetime if kept temporarily).
- [x] Outliers/implausible values (e.g., zero-guest bookings, negative `adr`) handled with a documented rule.
- [x] Stratified 80/20 split with `random_state=42`, test-set class proportion verified to match overall.
- [x] All transformers fit on training data only.
- [x] Processed splits saved to `data/processed/` with versioned names.
- [x] Preprocessing version label recorded (e.g., `prep_v1`) for the experiment log.


## Handoff To Notebook 04

- Frozen artifacts: `data/interim/hotel_cleaned_v1.csv`, `data/processed/X_train_v1.csv`, `data/processed/X_test_v1.csv`, `data/processed/y_train_v1.csv`, and `data/processed/y_test_v1.csv`.
- Notebook 04 (`04_feature_engineering.ipynb`) should start from the saved training split and apply any feature engineering in a train-aware way before mirroring the same logic onto the test split.
- Deferred decisions carried forward from this notebook: final keep/drop review for `assigned_room_type` and `booking_changes`, and final encoding treatment for `country`, `agent`, and `company`.
- Do not refit preprocessors downstream on the full dataset; preserve the fit-on-train/apply-on-test boundary documented above.
